In [21]:
# Look to see what kind of wins are happening.

import pandas as pd, numpy as np
# Import team data - the 'game_data' is misleading....
team_data = pd.read_csv('team_data_full.csv')
# Keep necessary columns
team_data = team_data[['year','team','wins','exp_win','rating','top_tier','era','type']]
# Upload individual game data, clean up
df = pd.read_csv('all_games.csv')
df = df[df['seasonType']=='regular'][['season','homeTeam','homePoints','homeClassification','homeConference','awayTeam','awayPoints','awayClassification','awayConference']]

# Rarrange data so each team has their own row - allows for easier aggregation
temp = df[['season','awayTeam','awayPoints','awayClassification','awayConference','homeTeam','homePoints','homeClassification','homeConference',]]
temp.columns = ['season','team','scored','class','conf','opponent','allowed','opp_class','opp_conf']

df.columns = ['season','team','scored','class','conf','opponent','allowed','opp_class','opp_conf']

df = pd.concat([df, temp], ignore_index=True)

# Define outcome
df['outcome'] = np.where(df['scored'] > df['allowed'], 'w', 'l')

# Merge to analyze types of wins
merged = df.merge(
    team_data,
    left_on=['season', 'team'],
    right_on=['year', 'team'],
    how='left'
)
merged = merged.drop(columns=['year'])


# Modify headers
id_cols = merged.columns[-7:]
merged = merged.rename(columns={col: 'team_' + col for col in id_cols})

# Repeat to get data for opponents
merged = merged.merge(
    team_data,
    left_on=['season', 'opponent'],
    right_on=['year', 'team'],
    how='left'
)
merged = merged.drop(columns=['year'])
merged = merged.rename(columns={col: 'opp_' + col for col in id_cols})
merged = merged.drop(columns=['team_y'])
merged = merged.rename(columns={'team_x': 'team'})

# Adjust classification for class and opp_class. Keep as FCS or lower if not in g4/p6 list. Go off of conference at time
# Define p4 and g5 to differentiate
p4 = ['SEC','Big 12','Big Ten','ACC','Pac-12']
g5 = ['Mountain West','Sun Belt','Mid-American','American Athletic','Conference USA']
# Do for team first
merged['class'] = merged.apply(
    lambda x: 'g5' if x['conf'] in g5
    else 'g5' if x['conf'] == 'pac-12' and x['season'] >= 2024
    else 'p4' if x['conf'] in p4
    else x['class'],
    axis=1
)
# Repeat for opposition
merged['opp_class'] = merged.apply(
    lambda x: 'g5' if x['opp_conf'] in g5
    else 'g5' if x['opp_conf'] == 'pac-12' and x['season'] >= 2024
    else 'p4' if x['opp_conf'] in p4
    else x['opp_class'],
    axis=1
)
# Account for Notre dame being a p4 school, otherwise change the g5

merged.loc[(merged['conf'] == 'FBS Independents') & (merged['team'] == 'Notre Dame'), 'class'] = 'p4'
merged.loc[(merged['opp_conf'] == 'FBS Independents') & (merged['opponent'] == 'Notre Dame'), 'opp_class'] = 'p4'
merged.loc[(merged['conf'] == 'FBS Independents') & (merged['team'] != 'Notre Dame'), 'class'] = 'g5'
merged.loc[(merged['opp_conf'] == 'FBS Independents') & (merged['opponent'] != 'Notre Dame'), 'opp_class'] = 'g5'

merged.to_csv("individual_games.csv",index=False)



/tmp/ipykernel_1403/512428275.py:9: DtypeWarning: Columns (0: highlights) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('all_games.csv')


In [22]:
# Make a w loss record for each era
import pandas as pd
df = pd.read_csv("individual_games.csv")


df = df[((df['class'] == 'p4' )| (df['class'] == 'g5' )) & (df['opp_class'].notna())]
df = df.groupby(['team_era', 'class', 'opp_class', 'team_outcome']).size().unstack(fill_value=0).reset_index()
df['games'] = (df['w'] + df['l'])
df['rate'] = df['w'] / df['games']
df = df.sort_values(['class','opp_class'])
print(df)

team_outcome team_era class opp_class     l     w  games      rate
0                post    g5       fcs    26   252    278  0.906475
6                 pre    g5       fcs    26   245    271  0.904059
1                post    g5        g5  1372  1321   2693  0.490531
7                 pre    g5        g5  1381  1431   2812  0.508890
2                post    g5        p4   373    75    448  0.167411
8                 pre    g5        p4   403    95    498  0.190763
3                post    p4       fcs     6   278    284  0.978873
9                 pre    p4       fcs    11   235    246  0.955285
4                post    p4        g5    87   430    517  0.831721
10                pre    p4        g5    98   443    541  0.818854
5                post    p4        p4  1655  1655   3310  0.500000
11                pre    p4        p4  1576  1576   3152  0.500000


In [23]:
# Look at total games and win record by conferece type to see how often the heavyweights beat eachother
import pandas as pd
df = pd.read_csv("individual_games.csv")

# df.columns
df = df[((df['class'] == 'p4' )| (df['class'] == 'g5' )) & (df['opp_class'].notna()) & ((df['team_wins'] >= 10)|  (df['team_exp_win'] >= 10)) & (df['opp_wins'] >= 10)]
df = df.groupby(['team_era', 'class', 'opp_class', 'team_outcome']).size().unstack(fill_value=0).reset_index()
df['games'] = (df['w'] + df['l'])
df['rate'] = df['w'] / df['games']
df = df.sort_values(['class','opp_class'])
print(df)

team_outcome team_era class opp_class   l   w  games      rate
0                post    g5        g5  16  15     31  0.483871
4                 pre    g5        g5  22  21     43  0.488372
1                post    g5        p4   8   4     12  0.333333
5                 pre    g5        p4  11   0     11  0.000000
2                post    p4        g5   4   8     12  0.666667
6                 pre    p4        g5   0  11     11  1.000000
3                post    p4        p4  68  65    133  0.488722
7                 pre    p4        p4  58  50    108  0.462963


In [24]:
# Look at individual games between heavyweights to see any trends
import pandas as pd
df = pd.read_csv("individual_games.csv")

df = df[((df['opp_class'] == 'p4' ) & (df['class'] == 'g5' )) & (((df['team_wins'] >= 10)|  (df['team_exp_win'] >= 10))) & (df['opp_wins'] >= 10)]
df = df[['team_era','season','team','team_wins','team_rating','team_outcome','opponent','opp_wins','opp_rating','scored','allowed']].sort_values(['season','team_outcome'])
df['margin'] = abs(df['scored'] - df['allowed'])
print(df.groupby('team_era')['margin'].mean())
df.head(len(df))



team_era
post    18.666667
pre     21.000000
Name: margin, dtype: float64


,team_era,season,team,team_wins,team_rating,team_outcome,opponent,opp_wins,opp_rating,scored,allowed,margin
1030,pre,2015,Temple,10.0,6.0,l,Notre Dame,10.0,22.4,20.0,24.0,4.0
29126,pre,2015,App State,10.0,7.9,l,Clemson,13.0,24.0,10.0,41.0,31.0
29625,pre,2015,Navy,10.0,12.1,l,Notre Dame,10.0,22.4,24.0,41.0,17.0
30636,pre,2016,Western Kentucky,10.0,15.5,l,Alabama,13.0,31.7,10.0,38.0,28.0
30731,pre,2016,Temple,10.0,11.9,l,Penn State,11.0,19.0,27.0,34.0,7.0
32117,pre,2017,Florida Atlantic,10.0,5.1,l,Wisconsin,12.0,26.8,14.0,31.0,17.0
32382,pre,2017,Toledo,11.0,6.6,l,Miami,10.0,17.4,30.0,52.0,22.0
33936,pre,2018,Army,10.0,-1.5,l,Oklahoma,12.0,24.9,21.0,28.0,7.0
35035,pre,2019,Florida Atlantic,10.0,-0.6,l,Ohio State,13.0,35.4,21.0,45.0,24.0
35125,pre,2019,Cincinnati,10.0,9.1,l,Ohio State,13.0,35.4,0.0,42.0,42.0


In [17]:
# Look at strength of schedule. See how often, in either era, a g5/p4 team played a solid opponent. WIll reference exp wins, total wins, and SP+ rating.
import pandas as pd
df = pd.read_csv("individual_games.csv")
df = df[((df['opp_class'] == 'p4' ) & (df['class'] == 'g5' )) & (df['team_wins'] >= 10) | (df['team_exp_win'] >= 10)]

# Define threshold, top 25% of each
threshold = 0.90
df['opp_wins_threshold'] = df.groupby('season')['opp_wins'].transform(lambda x: x.quantile(threshold))
df['opp_exp_win_threshold'] = df.groupby('season')['opp_exp_win'].transform(lambda x: x.quantile(threshold))
df['opp_rating_threshold'] = df.groupby('season')['opp_rating'].transform(lambda x: x.quantile(threshold))

result = df.groupby(['team_era', 'class']).agg(
    opp_wins_count=('opp_wins', lambda x: (x > df.loc[x.index, 'opp_wins_threshold']).values.sum()),
    opp_exp_win_count=('opp_exp_win', lambda x: (x > df.loc[x.index, 'opp_exp_win_threshold']).values.sum()),
    opp_rating_count=('opp_rating', lambda x: (x > df.loc[x.index, 'opp_rating_threshold']).values.sum()),
    total_games=('opp_wins', 'size')
).reset_index()

result['opp_wins_pct'] = (result['opp_wins_count'] / result['total_games'] * 100).round(1)
result['opp_exp_win_pct'] = (result['opp_exp_win_count'] / result['total_games'] * 100).round(1)
result['opp_rating_pct'] = (result['opp_rating_count'] / result['total_games'] * 100).round(1)

result = result.drop('total_games', axis=1).sort_values(['class'])
result.head()




,team_era,class,opp_wins_count,opp_exp_win_count,opp_rating_count,opp_wins_pct,opp_exp_win_pct,opp_rating_pct
0,post,g5,12,14,8,5.4,6.3,3.6
2,pre,g5,15,17,10,5.8,6.6,3.9
1,post,p4,47,59,69,8.0,10.0,11.7
3,pre,p4,48,58,61,8.3,10.0,10.5


In [16]:
# Repeat prior block, but only look at wins
import pandas as pd
df = pd.read_csv("individual_games.csv")
df = df[((df['opp_class'] == 'p4' ) & (df['class'] == 'g5' )) & (df['team_wins'] >= 10) | (df['team_exp_win'] >= 10) & (df['team_outcome'] == 'w')]

# Define threshold
threshold = 0.90
df['opp_wins_threshold'] = df.groupby('season')['opp_wins'].transform(lambda x: x.quantile(threshold))
df['opp_exp_win_threshold'] = df.groupby('season')['opp_exp_win'].transform(lambda x: x.quantile(threshold))
df['opp_rating_threshold'] = df.groupby('season')['opp_rating'].transform(lambda x: x.quantile(threshold))

result = df.groupby(['team_era', 'class']).agg(
    opp_wins_count=('opp_wins', lambda x: (x > df.loc[x.index, 'opp_wins_threshold']).values.sum()),
    opp_exp_win_count=('opp_exp_win', lambda x: (x > df.loc[x.index, 'opp_exp_win_threshold']).values.sum()),
    opp_rating_count=('opp_rating', lambda x: (x > df.loc[x.index, 'opp_rating_threshold']).values.sum()),
    total_games=('opp_wins', 'size')
).reset_index()

result['opp_wins_pct'] = (result['opp_wins_count'] / result['total_games'] * 100).round(1)
result['opp_exp_win_pct'] = (result['opp_exp_win_count'] / result['total_games'] * 100).round(1)
result['opp_rating_pct'] = (result['opp_rating_count'] / result['total_games'] * 100).round(1)

result = result.drop('total_games', axis=1).sort_values(['class'])
result

,team_era,class,opp_wins_count,opp_exp_win_count,opp_rating_count,opp_wins_pct,opp_exp_win_pct,opp_rating_pct
0,post,g5,13,19,10,6.5,9.5,5.0
2,pre,g5,14,17,13,6.0,7.2,5.5
1,post,p4,28,46,54,5.5,9.0,10.5
3,pre,p4,28,51,54,5.5,10.1,10.7


In [27]:
# Look at polarization of outcomes; see if more blowouts happened
import pandas as pd
df = pd.read_csv("individual_games.csv")
# Look at number of games decided by <7 points, 7-14, 14-21, 21+
df['margin'] = abs(df['scored'] - df['allowed'])
bins = [0, 7, 14, 21, float('inf')]
labels = ['<=7', '8-14', '15-21', '>=22']
df['margin_bin'] = pd.cut(df['margin'], bins=bins, labels=labels, right=True)

result = df.groupby(['team_era', 'class','opp_class', 'margin_bin']).size().unstack(fill_value=0).reset_index().sort_values(['class','opp_class','team_era'])
result = result[result['opp_class']!='fcs']
print("Table showing how often games were decided by a certain point threshold")
print(result)

# Repeat to show top teams vs other top teams
df_w = df[((df['opp_class'] == 'p4' ) & (df['class'] == 'g5' )) & (df['team_wins'] >= 10) | (df['team_exp_win'] >= 10) & (df['team_outcome'] == 'w')]
print("Does it show the table and this?")
result_w = df_w.groupby(['team_era', 'class','opp_class', 'margin_bin']).size().unstack(fill_value=0).reset_index().sort_values(['class','opp_class','team_era'])
result_w = result_w[result_w['opp_class']!='fcs']
print("Table showing how often games were decided by a certain point threshold, only looking at top teams")
print(result_w)


Table showing how often games were decided by a certain point threshold
margin_bin team_era class opp_class   <=7  8-14  15-21  >=22
1              post    g5        g5   987   542    474   689
7               pre    g5        g5   930   540    501   841
2              post    g5        p4   103    53     78   214
8               pre    g5        p4   122    71     68   237
4              post    p4        g5   123    62     86   246
10              pre    p4        g5   128    76     73   264
5              post    p4        p4  1230   650    500   930
11              pre    p4        p4  1142   558    524   928
Does it show the table and this?
Table showing how often games were decided by a certain point threshold, only looking at top teams
margin_bin team_era class opp_class  <=7  8-14  15-21  >=22
1              post    g5        g5   26    26     27    58
7               pre    g5        g5   25    24     33    82
2              post    g5        p4   16     8     14    10
8      

In [5]:
# Now we'll look to see if the 21-22 stretch had differences than the 23-25 stretch. We'll start off by counting the number of 10 win teams
import pandas as pd, numpy as np
df = pd.read_csv("individual_games.csv")
# Simplify data, only keep post NILP schools
post = df[df['team_era'] == 'post'][['season','team','class','conf','team_wins','team_exp_win','team_rating']]

result = post.groupby(['season', 'class']).agg(
    teams_10plus=('team_rating', lambda x: (x >= 10).sum()),
    total_teams=('team_exp_win', 'size')
).reset_index()

result['pct_10plus'] = (result['teams_10plus'] / result['total_teams'] * 100).round(1)
print("Looking at both conference types, post NILP")
print(result.sort_values(['class','season']))

# repeat, but look at all years for g5
g5 = df[df['class'] == 'g5'][['season','team','class','conf','team_exp_win','team_exp_win','team_rating']]

result_g5 = g5.groupby(['season', 'class']).agg(
    teams_10plus=('team_rating', lambda x: (x >= 10).sum()),
    total_teams=('team_rating', 'size')
).reset_index()

result_g5['pct_10plus'] = (result_g5['teams_10plus'] / result_g5['total_teams'] * 100).round(1)
print("all years, just g5 schools")
print(result_g5.sort_values(['class','season']))


Looking at both conference types, post NILP
   season class  teams_10plus  total_teams  pct_10plus
0    2021    g5            64          719         8.9
2    2022    g5            50          718         7.0
4    2023    g5            13          669         1.9
6    2024    g5            13          658         2.0
8    2025    g5            25          655         3.8
1    2021    p4           406          790        51.4
3    2022    p4           392          787        49.8
5    2023    p4           271          838        32.3
7    2024    p4           355          848        41.9
9    2025    p4           355          848        41.9
all years, just g5 schools
    season class  teams_10plus  total_teams  pct_10plus
0     2015    g5            37          765         4.8
1     2016    g5            88          765        11.5
2     2017    g5            23          783         2.9
3     2018    g5            62          790         7.8
4     2019    g5            50          792 